In [1]:
import torch

`import torch`

This statement imports the **torch** package (PyTorch) into the current Python namespace under the name `torch`. PyTorch is a widely used open-source library for tensor computation with GPU acceleration and for building and training neural networks.

After this import, you can reference classes, functions, and submodules such as:

- `torch.Tensor`
- `torch.nn`
- `torch.optim`
- `torch.autograd`
- and others.

**Conceptually:**  
`torch` provides an n-dimensional array object (`torch.Tensor`) and automatic differentiation (`autograd`). Autograd constructs a computational graph of tensor operations; when `.backward()` is called on a scalar-valued tensor, gradients of that scalar with respect to any intermediate tensors with `requires_grad=True` are computed via **reverse-mode automatic differentiation**.

In [2]:
class minimize_quadratic(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.w = torch.nn.Parameter(data=torch.tensor([0.0]))

    def forward(self):

        g = self.w
        h = (g-6)
        i = h**2
        j = 8*i
        f = (j+9)

        return f

`class minimize_quadratic(torch.nn.Module):`

**Explanation:**

This defines a new Python class named `minimize_quadratic` that inherits from `torch.nn.Module`.

`torch.nn.Module` is the base class for all neural-network modules in PyTorch. Subclassing `torch.nn.Module` gives the new class:

- A standardized way to hold **parameters** (tensors that should be optimized).  
- A `parameters()` method that yields parameters for optimizers.  
- The convention of implementing a `.forward(...)` method that defines computation of the value of the function at a specific value of input. 

**Design intent:**  
We are modelling a scalar variable `w` (a parameter) and defined a `forward` method that computes the value of a quadratic function at `w`, allowing an optimizer to minimize it.

Our quadratic function is:

- $f(w) = 8\cdot(w - 6)^2 + 9$

`def __init__(self):`

**Explanation:**

This begins the constructor method for the class. When `minimize_quadratic()` is called or object of this class is created, Python automatically calls the `__init__` method to initialize the instance and set up any attributes or parameters required by the class.

`super().__init__()`

**Explanation:**

This calls the constructor of the base class (`torch.nn.Module.__init__`). It is required to properly initialize the internal machinery of `Module`, such as the registration of parameters, buffers, and submodules.

If `super().__init__()` is not called, essential internal attributes (such as `_parameters` and `_modules`) would not be initialized correctly, and methods like `parameters()`, `to(device)`, or state management utilities could fail.

`self.w = torch.nn.Parameter(data=torch.tensor([0.0]))`

**Explanation:**

- Creates a new tensor `torch.tensor([0.0])`, which is a 1-dimensional tensor containing a single float value `0.0`.  
- Wraps that tensor in `torch.nn.Parameter` and assigns it to `self.w`.

**Why `Parameter` rather than a plain `Tensor`:**

- `torch.nn.Parameter` is a subclass of `torch.Tensor` that is automatically added to the module’s set of parameters when assigned as an attribute of an `nn.Module` (such as `self.w`).  
- As a result, any optimizer (such as `SGD` or `Adagrad` or `RMSProp` etc) created using `model.parameters()` will include `self.w` and can update it's initial guess.
- `torch.nn.Parameter` defaults to `requires_grad=True`, which tracks all operations happening in the forward pass involving this value and compute gradients  
  $\nabla_w\cdot{f}$  
  when `.backward()` is called.

**Shape and dtype details:**

- The tensor has shape `(1,)`, so `self.w` is a single-element parameter.  
- Extracting the scalar later may require indexing or `.item()`, e.g., `self.w[0]` or `self.w.item()`.  
- If GPU computation is needed, moving the module with `.to(device)` will also move `self.w` automatically.

**Mathematical significance:**

- `w` is the optimization variable.  
- For a function, $f(w) = 8\cdot{(w-6)^2} + 9$,  
  the gradient is  
  $\nabla_w\cdot{f} = 16\cdot{(w-6)}$.


`def forward(self):`

**Explanation:**

This declares the `forward` method. In `torch.nn.Module`, `forward` defines the computation executed when the module instance is called. When you invoke `nn()` on a module object, PyTorch internally routes the call to `nn.forward()`.

**Typical role here:**

Given the current parameter `self.w`, this method computes and returns the function value $f(w)$ that we want to minimize.  
For our quadratic function, the implementation inside `forward` function is:

```
g = self.w
h = (g-6)
i = h**2
j = 8*i
f = (j+9)

return f
```

Even without external input tensors, a module may define `forward(self)` using only its internal parameters. This is common in toy examples where the function consists of a single trainable scalar which is exactly our case. 

**Mathematical note:**

The method will compute:

$f(w) = 8\cdot{(w-6)^2} + 9$

The optimizer will minimize $f(w)$, and gradient-based methods (`SGD` or `Adagrad` or `RMSProp` etc) will use the derivative of this expression with respect to $w$.


In [14]:
nn = minimize_quadratic()

In [15]:
optimizer = torch.optim.SGD(params=nn.parameters(),lr=0.01)
tol = 10**(-5)
epoch = 0
nn_params = list()

while(True):

    initial_function_value = nn()
    optimizer.zero_grad()
    initial_function_value.backward()
    optimizer.step()
    final_function_value = nn()

    if torch.abs(final_function_value - initial_function_value) < tol:
        break

    epoch += 1
    print("Epoch # {}, Function Value = {}".format(epoch, initial_function_value[0]))


for params in nn.parameters():
    nn_params.append(params.detach().numpy())


print("\n\nThe Global Minima of the Function is {}".format(nn_params[0][0]))

Epoch # 1, Function Value = 297.0
Epoch # 2, Function Value = 212.21279907226562
Epoch # 3, Function Value = 152.386962890625
Epoch # 4, Function Value = 110.17383575439453
Epoch # 5, Function Value = 80.38825988769531
Epoch # 6, Function Value = 59.371551513671875
Epoch # 7, Function Value = 44.54216384887695
Epoch # 8, Function Value = 34.078556060791016
Epoch # 9, Function Value = 26.695432662963867
Epoch # 10, Function Value = 21.485898971557617
Epoch # 11, Function Value = 17.810047149658203
Epoch # 12, Function Value = 15.216367721557617
Epoch # 13, Function Value = 13.386269569396973
Epoch # 14, Function Value = 12.094950675964355
Epoch # 15, Function Value = 11.183799743652344
Epoch # 16, Function Value = 10.540887832641602
Epoch # 17, Function Value = 10.087250709533691
Epoch # 18, Function Value = 9.76716423034668
Epoch # 19, Function Value = 9.541311264038086
Epoch # 20, Function Value = 9.381948471069336
Epoch # 21, Function Value = 9.269503593444824
Epoch # 22, Function Va

**Model Creation:**

`nn = minimize_quadratic()`

This creates an instance of the model.


**Optimizer creation:**

A typical optimizer setup:

`optimizer = torch.optim.SGD(params=nn.parameters(), lr=0.01)`

`torch.optim.SGD` performs stochastic gradient descent.
`params=nn.parameters()` specifies that the optimizer should update the model parameters (here, self.w).
lr is the learning rate $\epsilon$, a positive scalar that controls the step size.


```
tol = 10**(-5)
epoch = 0
nn_params = list()

while(True):

    initial_function_value = nn()
    optimizer.zero_grad()
    initial_function_value.backward()
    optimizer.step()
    final_function_value = nn()

    if torch.abs(final_function_value - initial_function_value) < tol:
        break

    epoch += 1
    print("Epoch # {}, Function Value = {}".format(epoch, initial_function_value[0]))


for params in nn.parameters():
    nn_params.append(params.detach().numpy())

print("\n\nThe Global Minima of the Function is {}".format(nn_params[0][0]))
```

**Mathematical explanation:**

For gradient descent on a scalar parameter $w$ with learning rate $\epsilon$, the update rule is:

$w_\text{final} = w_\text{initial} - \epsilon \cdot{\nabla_w f(w_\text{initial})}$

Our quadratic function is:

$f(w) = 8(w - 6)^{2} + 9$

then the derivative is:

$\nabla_w\cdot{f} = 16(w - 6)$.

Gradient descent converges linearly to the minimum when $\epsilon$ is sufficiently small to satisfy stability requirements.

**Convergence check:**

The condition:

```
if torch.abs(final_function_value - initial_function_value) < tol:
    break
```


terminates the iteration when the improvement between two successive function evaluations, one at initial value of $w$ and another at final value of $w$, becomes smaller than the tolerance `tol`.

Mathematically, the stopping rule is:

$|f(w_\text{final}) - f(w_\text{initial})| < \text{tol}$   

a standard convergence criterion in iterative optimization.